# MCSDCA x LeWM PushT - Results

All five LeWM modules (`encoder`, `projector`, `action_encoder`, `predictor`,
`pred_proj`) are trained jointly from scratch on `prediction MSE + sigreg_weight * SIGReg`.

Every optimizer gets the same initial weights, the same batch stream, and the same
**backprop budget** (number of backward passes), so results are compared on the
`backprop_calls` axis - one MCSDCA outer step performs several backward passes.

Each run writes just `metrics.csv` (one row per evaluation) and `run.json`
(args + profile + resolved MCSDCA config + final metrics). Set `RUN_DIR` to pin a
run, or leave it `None` for the newest one under `outputs/pusht_predictor_optimizer/`.


In [ ]:
from __future__ import annotations

import csv
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "run_pusht_predictor_experiment.py").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Cannot find repository root.")

# Pin a specific result directory, or leave None for the newest.
RUN_DIR = None
OUTPUT_ROOT = ROOT / "outputs" / "pusht_predictor_optimizer"
ABLATION_ROOT = ROOT / "outputs" / "ablation"


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def read_csv(path: Path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def is_run_dir(path: Path) -> bool:
    return path.is_dir() and (path / "run.json").exists() and (path / "metrics.csv").exists()


def latest_run(root: Path) -> Path:
    runs = [p for p in root.iterdir() if is_run_dir(p)] if root.exists() else []
    if not runs:
        raise FileNotFoundError(f"No run.json/metrics.csv run found under {root}.")
    return max(runs, key=lambda p: p.stat().st_mtime)


run_dir = Path(RUN_DIR).resolve() if RUN_DIR else latest_run(OUTPUT_ROOT)
display(Markdown(f"Run: `{run_dir}`"))


In [ ]:
run_info = read_json(run_dir / "run.json")
metrics = read_csv(run_dir / "metrics.csv")


def num(row, key):
    value = row.get(key, "")
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def fmt(value):
    if value in (None, ""):
        return ""
    try:
        x = float(value)
    except (TypeError, ValueError):
        return str(value)
    if x == 0:
        return "0"
    return f"{x:.3e}" if abs(x) >= 1000 or abs(x) < 1e-3 else f"{x:.6f}"


def table(rows, columns):
    head = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(fmt(r.get(c, "")) for c in columns) + " |" for r in rows]
    return "\n".join([head, sep, *body])


by_optimizer = defaultdict(list)
for row in metrics:
    by_optimizer[row["optimizer"]].append(row)
for rows in by_optimizer.values():
    rows.sort(key=lambda r: num(r, "backprop_calls") or 0)

final_rows = [rows[-1] for rows in by_optimizer.values()]
display(Markdown(f"**{len(by_optimizer)} optimizers**, "
                 f"budget={run_info.get('budget')} (~{run_info.get('epochs_equivalent', 0):.1f} epochs), "
                 f"profile=`{run_info.get('profile', {}).get('name')}` "
                 f"data_fraction={run_info.get('profile', {}).get('data_fraction')}"))


In [ ]:
args = run_info["args"]
profile = run_info.get("profile", {})
mcsdca = run_info.get("mcsdca_config", {})

config_rows = [
    {"key": "profile", "value": profile.get("name")},
    {"key": "data fraction", "value": profile.get("data_fraction")},
    {"key": "train windows", "value": run_info.get("train_windows")},
    {"key": "backprop budget", "value": run_info.get("budget")},
    {"key": "steps / epoch", "value": run_info.get("steps_per_epoch")},
    {"key": "epochs equivalent", "value": run_info.get("epochs_equivalent")},
    {"key": "batch size", "value": profile.get("batch_size")},
    {"key": "precision", "value": profile.get("precision")},
    {"key": "SIGReg projections", "value": profile.get("sigreg_num_proj")},
    {"key": "objective", "value": f"pred_mse + {args.get('sigreg_weight')} * SIGReg"},
    {"key": "baseline lr / wd", "value": f"{args.get('lr')} / {args.get('weight_decay')}"},
    {"key": "device", "value": run_info.get("device")},
    {"key": "seed", "value": args.get("seed")},
]
display(Markdown("### Run configuration\n\n" + table(config_rows, ["key", "value"])))

mcsdca_rows = [{"parameter": k, "value": v} for k, v in mcsdca.items()]
display(Markdown("### Resolved MCSDCA config\n\n" + table(mcsdca_rows, ["parameter", "value"])))


In [ ]:
predictor_cols = ["optimizer", "backprop_calls", "train_mse", "val_mse", "train_val_gap",
                  "latent_norm_drift", "pred_latent_variance", "train_time_s", "status"]
rollout_cols = ["optimizer", "rollout_mse_1", "rollout_mse_3", "rollout_mse_5",
                "latent_norm_drift", "target_latent_norm", "status"]
order = ["AdamW", "Adam", "SGD + momentum", "RMSprop", "Adagrad", "MCSDCA-odLD", "MCSDCA-udLD"]
final_sorted = sorted(final_rows, key=lambda r: order.index(r["optimizer"]) if r["optimizer"] in order else 99)

display(Markdown("### Final one-step prediction\n\n" + table(final_sorted, predictor_cols)))
display(Markdown("### Final multi-step latent rollout\n\n" + table(final_sorted, rollout_cols)))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for optimizer, rows in by_optimizer.items():
    x = [num(r, "backprop_calls") for r in rows]
    axes[0].plot(x, [num(r, "train_mse") for r in rows], marker="o", label=f"{optimizer} train")
    axes[0].plot(x, [num(r, "val_mse") for r in rows], marker="x", linestyle="--", label=f"{optimizer} val")
    axes[1].plot(x, [num(r, "rollout_mse_5") for r in rows], marker="o", label=optimizer)
    axes[2].plot(x, [num(r, "latent_norm_drift") for r in rows], marker="o", label=optimizer)
for ax, title, ylabel in zip(
    axes,
    ["One-step MSE", "Rollout MSE@5", "Latent norm drift"],
    ["MSE", "MSE", "pred_norm - target_norm"],
):
    ax.set_title(title)
    ax.set_xlabel("backprop calls")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
tail_cols = ["optimizer", "backprop_calls", "outer_step", "learning_rate", "train_mse",
             "val_mse", "rollout_mse_5", "latent_norm_drift", "markov_chain_length",
             "retained_samples", "gamma_k", "sampler_loss", "train_time_s"]
tail = []
for rows in by_optimizer.values():
    tail.extend(rows[-4:])
display(Markdown("### Last evaluations per optimizer\n\n" + table(tail, tail_cols)))


In [ ]:
# Data-scaling ablation: newest outputs/ablation/<ts>/ablation.csv
abl_dirs = sorted((p for p in ABLATION_ROOT.glob("*/ablation.csv")), key=lambda p: p.stat().st_mtime) if ABLATION_ROOT.exists() else []
if not abl_dirs:
    display(Markdown("_No ablation run found under `outputs/ablation/`. Run `src/run_ablation.py`._"))
else:
    abl = read_csv(abl_dirs[-1])
    display(Markdown(f"Ablation: `{abl_dirs[-1].parent.name}`"))
    metrics_to_plot = ["val_mse", "rollout_mse_5", "train_val_gap"]
    optimizers = sorted({r["optimizer"] for r in abl})
    fractions = sorted({float(r["data_fraction"]) for r in abl})
    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(6 * len(metrics_to_plot), 4.5))
    for ax, metric in zip(axes, metrics_to_plot):
        for optimizer in optimizers:
            means, stds = [], []
            for frac in fractions:
                vals = [float(r[metric]) for r in abl
                        if r["optimizer"] == optimizer and float(r["data_fraction"]) == frac
                        and r.get("status") == "ok" and r.get(metric) not in (None, "")]
                means.append(sum(vals) / len(vals) if vals else float("nan"))
                stds.append((sum((v - means[-1]) ** 2 for v in vals) / len(vals)) ** 0.5 if len(vals) > 1 else 0.0)
            ax.errorbar(fractions, means, yerr=stds, marker="o", capsize=3, label=optimizer)
        ax.set_title(metric)
        ax.set_xlabel("data fraction")
        ax.set_xscale("log")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## Reading the results

- Compare optimizers at equal `backprop_calls`; one MCSDCA outer step is several backward passes.
- `train_mse` / `val_mse` are prediction-only; the optimized objective also includes SIGReg.
- `status = diverged` means that optimizer hit a non-finite loss and was skipped; its
  row carries the `error` and the other optimizers still completed.
- A small train/val gap alone does not imply useful control - watch rollout MSE, the
  latent-norm drift, and (separately) `evaluate_planning.py` PushT success.
- The ablation asks whether MCSDCA's advantage grows when data is scarce (small
  fraction = more passes over less data = more overfitting pressure).
